In [1]:
!pip3 install pulp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.7/17.7 MB 3.7 MB/s eta 0:00:00m eta 0:00:010:00:01


In [2]:
import os

class Graph:
    def __init__(self, n):
        self.edgeList = []
        self.adj_list = []
        for i in range(0, n + 1):
            self.adj_list.append([])

    def add_edge(self, u, v):
        self.adj_list[u].append(v)
        self.adj_list[v].append(u)

        self.edgeList.append((v,u))


def read_edges_from_file(filename):
    with open(filename, 'r') as file:
        edges = [tuple(map(int, line.strip().split())) for line in file]
    return edges[0][0], edges[1:]

def create_graph_from_edges(edges, n):
    graph = Graph(n)
    for u, v in edges:
        graph.add_edge(u, v)
    return graph

In [8]:
def main():
    folder_path = "../generatedgraphs"  # Update this with the path to your folder
    graph_files = [file for file in os.listdir(folder_path) if file.endswith('.txt')]

    graphs = []
    for file in graph_files:
        n, edges = read_edges_from_file(os.path.join(folder_path, file))
        graph = create_graph_from_edges(edges, n)  # Replace n with the appropriate value
        graphs.append(graph)

    return graphs

In [9]:
g = main()

In [10]:
class GraphDrawConfiguration:
    with_labels: bool = False
    font_weight: str = 'bold'
    node_color: str = 'red'
    edge_color: str = 'black'
    node_size: int = 1


conf = GraphDrawConfiguration()

In [13]:
import networkx as nx
import matplotlib.pyplot as plt

def save_as_image():
    for i in range(len(g)):

        edges = g[i].edgeList


        G = nx.Graph()
        G.add_edges_from(edges)

        nx.draw(G, with_labels=conf.with_labels,
                font_weight=conf.font_weight,
                node_color=conf.node_color,
                edge_color=conf.edge_color,
                node_size=conf.node_size
        )

        plt.savefig(f"/content/images/g[{i}]th graph")
        plt.close()

In [6]:
!zip -r images.zip images

  adding: images/ (stored 0%)
  adding: images/g[6]th graph.png (deflated 12%)
  adding: images/g[0]th graph.png (deflated 10%)
  adding: images/g[2]th graph.png (deflated 13%)
  adding: images/g[3]th graph.png (deflated 11%)
  adding: images/g[1]th graph.png (deflated 12%)
  adding: images/g[7]th graph.png (deflated 12%)
  adding: images/g[9]th graph.png (deflated 12%)
  adding: images/g[4]th graph.png (deflated 12%)
  adding: images/g[8]th graph.png (deflated 12%)
  adding: images/g[5]th graph.png (deflated 11%)


In [7]:
!rm -rf images images.zip
!mkdir images

In [14]:
def get_distance_matrix(g : Graph):
    G = nx.Graph()
    G.add_edges_from(g.edgeList)
    return nx.floyd_warshall_numpy(G)

In [16]:
from pulp import LpProblem, LpMinimize, LpVariable


def ILP_Solver(V, k, dist):
    problem = LpProblem("Packing Coloring Problem", LpMinimize)
    x = LpVariable.dicts("x", [(v, i) for v in V for i in range(1, k+1)], lowBound=0, upBound=1)
    z = LpVariable("z", lowBound=0)


    problem += z, "Objective function"

    # Constraints
    # Fractional coloring constraint
    for v in V:
        problem += sum(x[(v, i)] for i in range(1, k+1)) <= 1, f"Vertex {v} Coloring"


    # Adjacent coloring constraint
    for v in V:
        for u in V:
            for i in range(1, k+1):
                if dist[v-1][u-1] <= i:
                    problem += x[(v, i)] + x[(u, i)] <= 1, f"Adjacent Coloring {v}, {u}, {i}"

    # Coloring constraint
    for v in V:
        for i in range(1, k+1):
            problem += i * x[(v, i)] <= z, f"Coloring Constraint {v}, {i}"


    problem.solve()

    solution = {}
    for v in V:
        for i in range(1, k+1):
            if x[(v, i)].varValue > 0:
                solution[v] = i
                break

    return solution

In [17]:
graph = g[0]
n : int = len(graph.adj_list) - 1
V = [i for i in range(1, n + 1)]
k = n
dist = get_distance_matrix(graph)

In [18]:
G = nx.Graph()
G.add_edges_from(graph.edgeList)

In [ ]:
from rich import print

adj_list = nx.to_dict_of_lists(G)

# Print adjacency list
for node, neighbors in adj_list.items():
    print(f"{node}: {neighbors}")

In [20]:
dist.shape

(73, 73)

In [21]:
sol = ILP_Solver(V, k, dist)

/Users/theroyakash/Developer/GitHub/packing-coloring/RandomDegreeBoundedGraphs/.venv/lib/python3.12/site-packages/pulp/pulp.py:1316: UserWarning: Spaces are not permitted in the name. Converted to '_'
  warnings.warn("Spaces are not permitted in the name. Converted to '_'")


OSError: [Errno 86] Bad CPU type in executable: '/Users/theroyakash/Developer/GitHub/packing-coloring/RandomDegreeBoundedGraphs/.venv/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc'

In [23]:
sol

{}

In [22]:
from pulp import LpProblem, LpMinimize, LpVariable

def solve_LP_relaxation(V, k, d):
    # Initialize LP problem
    problem = LpProblem("LP Relaxation Coloring", LpMinimize)

    # Define decision variables
    x = LpVariable.dicts("x", [(v, i) for v in V for i in range(1, k+1)], cat="Binary")
    z = LpVariable("z", lowBound=0, cat="Integer")

    # Objective function: minimize z
    problem += z, "Objective Function"

    # Constraints
    # Fractional coloring constraint
    for v in V:
        problem += sum(x[(v, i)] for i in range(1, k+1)) == 1, f"Vertex {v} Coloring"

    # Adjacent coloring constraint
    for v in V:
        for u in V:
            for i in range(1, k+1):
                if d[v][u] <= i:
                    problem += x[(v, i)] + x[(u, i)] <= 1, f"Adjacent Coloring {v}, {u}, {i}"

    # Coloring constraint
    for v in V:
        for i in range(1, k+1):
            problem += i * x[(v, i)] <= z, f"Coloring Constraint {v}, {i}"

    # Solve the LP relaxation problem
    problem.solve()

    # Extract solution
    solution = {}
    for v in V:
        for i in range(1, k+1):
            print(f"x[{(v, i)}] = {x[(v,i)].value()}")
            if x[(v, i)].value() == 1:
                solution[v] = i

    return solution

# Example usage
V = ["A", "B", "C"]
k = 3
d = {
    "A": {"A": 0, "B": 1, "C": 2},
    "B": {"A": 1, "B": 0, "C": 1},
    "C": {"A": 2, "B": 1, "C": 0}
}

solution = solve_LP_relaxation(V, k, d)
print("Assigned colors:", solution)

OSError: [Errno 86] Bad CPU type in executable: '/Users/theroyakash/Developer/GitHub/packing-coloring/RandomDegreeBoundedGraphs/.venv/lib/python3.12/site-packages/pulp/solverdir/cbc/osx/64/cbc'